In [1]:
# Xây dựng mô hình GoogLeNet
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
class Inception(nn.Module):
    def __init__(self, in_channels, c1, c3red, c3, c5red, c5, pool):
        super().__init__()

        # Nhánh 1: Conv 1x1
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=1),
            nn.ReLU(inplace=True)
        )

        # Nhánh 2: Conv 1x1 -> Conv 3x3
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, c3red, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(c3red, c3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        # Nhánh 3: Conv 1x1 -> Conv 5x5
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, c5red, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(c5red, c5, kernel_size=5, padding=2),
            nn.ReLU(inplace=True)
        )

        # Nhánh 4: MaxPool -> Conv 1x1
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool, kernel_size=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)

        return torch.cat([b1, b2, b3, b4], dim=1)

In [3]:
class GoogLeNet(nn.Module):
    def __init__(self, num_classes=21):
        super().__init__()

        # Các lớp đầu
        # Nhiệm vụ trích xuất những đặc trưng ban đầu trước khi đưa vào Inception Block.
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            nn.Conv2d(64, 64, kernel_size=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # 2 Inception modules
        self.inception3a = Inception(192, 64, 96, 128, 16, 32, 32)
        self.inception3b = Inception(256, 128, 128, 192, 32, 96, 64)

        self.maxpool3 = nn.MaxPool2d(
            kernel_size=3, stride=2, padding=1
        )

        # 5 Inception modules
        self.inception4a = Inception(480, 192, 96, 208, 16, 48, 64)
        self.inception4b = Inception(512, 160, 112, 224, 24, 64, 64)
        self.inception4c = Inception(512, 128, 128, 256, 24, 64, 64)
        self.inception4d = Inception(512, 112, 144, 288, 32, 64, 64)
        self.inception4e = Inception(528, 256, 160, 320, 32, 128, 128)

        self.maxpool4 = nn.MaxPool2d(
            kernel_size=3, stride=2, padding=1
        )

        # 2 Inception modules
        self.inception5a = Inception(832, 256, 160, 320, 32, 128, 128)
        self.inception5b = Inception(832, 384, 192, 384, 48, 128, 128)

        # Global Average Pooling và Fully Connected
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.stem(x)

        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)

        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        x = self.inception4e(x)
        x = self.maxpool4(x)

        x = self.inception5a(x)
        x = self.inception5b(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)

        return x

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = GoogLeNet(num_classes=21).to(device)

print(model)

# Kiểm tra output
with torch.no_grad():
    x = torch.randn(2, 3, 224, 224).to(device)
    output = model(x)

print("Output shape:", output.shape)

GoogLeNet(
  (stem): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (4): ReLU(inplace=True)
    (5): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (inception3a): Inception(
    (branch1): Sequential(
      (0): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU(inplace=True)
    )
    (branch2): Sequential(
      (0): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(96, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
    (branch3): Sequential(
      (0): Conv2d(192, 16, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU(inplace=Tru

In [6]:
# Chuẩn bị dữ liệu VinaFood21
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Đổi sang đường dẫn dataset của bạn
DATA_DIR = r"C:\Users\Thao\Downloads\VinaFood21\VinaFood21"

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # một kỹ thuật Data Augmentation
    # Nếu chỉ học ảnh gốc, model có thể ghi nhớ một số bố cục cụ thể.
    # Khi dùng RandomHorizontalFlip, model tiếp xúc với nhiều cách bố trí khác nhau.
    # Giúp mô hình tổng quát hóa tốt hơn và giảm overfitting.
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

train_folder = os.path.join(DATA_DIR, "train")
val_folder = os.path.join(DATA_DIR, "val")

# Hai bản dataset dùng cùng ảnh nhưng transform khác nhau
full_train_aug = datasets.ImageFolder(
    train_folder, transform=train_transform
)

full_train_eval = datasets.ImageFolder(
    train_folder, transform=eval_transform
)

test_ds = datasets.ImageFolder(
    val_folder, transform=eval_transform
)

assert len(full_train_aug.classes) == 21
assert full_train_aug.class_to_idx == test_ds.class_to_idx

# Chia 90% train và 10% validation
n_train = int(0.9 * len(full_train_aug))
n_val = len(full_train_aug) - n_train

generator = torch.Generator().manual_seed(42)
# torch.Generator(): Tạo một bộ sinh số ngẫu nhiên (random number generator).
# generator này kiểm soát kết quả xáo trộn, mỗi lần chạy lại code, nhận được cùng một thứ tự.
indices = torch.randperm(
    len(full_train_aug), generator=generator
).tolist()

train_indices = indices[:n_train]
val_indices = indices[n_train:]

train_ds = torch.utils.data.Subset(
    full_train_aug, train_indices
)

val_ds = torch.utils.data.Subset(
    full_train_eval, val_indices
)
# Subset() để lấy một phần ảnh từ dataset gốc dựa trên danh sách indices.

batch_size = 16

train_dl = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True
)

val_dl = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False
)

test_dl = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False
)

classes = full_train_aug.classes

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))
print("Classes:", classes)

Train: 9039
Validation: 1005
Test: 6682
Classes: ['banh-can', 'banh-hoi', 'banh-mi-chao', 'banh-tet', 'banh-trang-tron', 'banh-u', 'banh-uot', 'bap-nuong', 'bo-kho', 'bo-la-lot', 'bot-chien', 'ca-ri', 'canh-kho-qua', 'canh-khoai-mo', 'ga-nuong', 'goi-ga', 'ha-cao', 'hoanh-thanh-nuoc', 'pha-lau', 'tau-hu', 'thit-kho-trung']


In [7]:
# Huấn luyện bằng Adam
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

EPOCHS = 20

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

In [9]:
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

In [ ]:
train_losses = []
val_losses = []
train_accs = []
val_accs = []

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model, train_dl, criterion, optimizer, device
    )

    val_loss, val_acc = validate(
        model, val_dl, criterion, device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            model.state_dict(),
            "best_googlenet.pth"
        )

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

C:\Users\Thao\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [1/20] | Train Loss: 2.9846 | Train Acc: 0.0881 | Val Loss: 2.9925 | Val Acc: 0.0905


In [ ]:
# Đánh giá Accuracy, Precision, Recall và F1-score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

model.load_state_dict(
    torch.load(
        "best_googlenet.pth",
        map_location=device,
        weights_only=True
    )
)

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_dl:
        images = images.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-macro:  {f1:.4f}")

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(21)),
    target_names=classes,
    zero_division=0
))

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GoogLeNet - Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("GoogLeNet - Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=list(range(21)),
    display_labels=classes,
    cmap="Blues",
    xticks_rotation=90,
    ax=ax
)

plt.title("GoogLeNet - VinaFood21 Confusion Matrix")
plt.tight_layout()
plt.show()